In [1]:
from gurobipy import *
import pandas as pd 
import numpy as np 

mod = Model('Bus Optimization')

club_filename = "/voc/data/club_data.xlsx"
routing_filename = "/voc/data/routing_solutions.csv"

alpha = 3.75

orig_avg_service_time = 18.43

beta = alpha / 4

Restricted license - for non-production use only - expires 2026-11-23


Use the mod object implemented above, along with correct (rounded) answers from PA 2, to implement Model 1 with beta = alpha / 4 in the following cell.

In [2]:
### Your code here
# read data
capacities = [14, 15, 24, 30, 66] # k
club_df = pd.read_excel(club_filename, header=1)
route_df = pd.read_csv(routing_filename, header=1)
club_df.columns = [str(i) for i in club_df.columns]
route_df.columns = [str(i) for i in route_df.columns]
club_names = club_df["Club"].tolist() # c
clubs = {}
for _, row in club_df.iterrows():
    club = row["Club"]
    clubs[club] = {
        "students": int(row["Students"]), # s_c
        "orig": {i: int(row[str(i)]) for i in capacities} # b_c,j for section 2
    }
total_students = sum(clubs[i]["students"] for i in club_names)
fleet = {j: sum(clubs[i]["orig"][j] for i in club_names) for j in capacities} # b_j
routes = {}
for _, row in route_df.iterrows():
    r = int(row["ID"])
    club = row["Club"]
    routes[r] = {
        "club": club, # c(r)
        "req": {k: int(row[str(k)]) for k in capacities}, # a_r,k
        "time": float(row["Avg Srvc Time"]) # t_r
    }
route_ids = list(routes.keys()) # r
routes_by_club = {c:[] for c in club_names} # r_c
for r in route_ids:
    routes_by_club[routes[r]["club"]].append(r)

# Decision variables
x = mod.addVars(route_ids, vtype=GRB.BINARY, name="x")
y = mod.addVars(club_names, capacities, vtype=GRB.INTEGER, name="y")
z = mod.addVar(vtype=GRB.CONTINUOUS, name="z")

# Objective. Minimize worst case capacity loss
mod.setObjective(z, GRB.MINIMIZE)

# Constraints
# Each club must have exactly one routing solution
for c in club_names:
    mod.addConstr(quicksum(x[r] for r in routes_by_club[c]) == 1)
    
# Each club must have >= number of buses of the capacity required by the routing solution
for c in club_names:
    for k in capacities:
        mod.addConstr(quicksum(y[c,j] for j in capacities if j >= k) >= quicksum(sum(routes[r]["req"][h] for h in capacities if h >= k) * x[r]
        for r in routes_by_club[c]))

# All buses of each capacity are assigned
for j in capacities:
    mod.addConstr(quicksum(y[c,j] for c in club_names) == fleet[j])
    
# Worst case seat loss constraint
for c in club_names:
    orig_capacity = sum(j * clubs[c]["orig"][j] for j in capacities)
    new_capacity = quicksum(j * y[c,j] for j in capacities)
    mod.addConstr(orig_capacity - new_capacity <= z)

# Average individual service time must improve by at least beta minutes
mod.addConstr(quicksum(clubs[c]["students"] * orig_avg_service_time for c in club_names) - quicksum(clubs[routes[r]["club"]]["students"] * routes[r]["time"] * x[r] for r in route_ids) >= total_students * beta)

# All variables non negative
for c in club_names:
    for j in capacities:
        mod.addConstr(y[c,j] >= 0)
mod.addConstr(z >= 0)

mod.optimize()

if mod.status == GRB.OPTIMAL:
    print("Optimal average service time:", mod.objVal)

# Section 3
if mod.status == GRB.OPTIMAL:
    avg_service_time = sum(clubs[routes[r]["club"]]["students"] * routes[r]["time"] * x[r].X for r in route_ids)/total_students
    print("Optimal worst case seat loss:", z.X)
    print("Average individual service time:", avg_service_time)
    
    print("\nBus assignments by club:")
    for c in club_names:
        vals = {j: int(round(y[c,j].X)) for j in capacities}
        print(c, vals)

    print("\nSelected routes:")
    for r in route_ids:
        if x[r].X > 0.5:
            print(r, routes[r]["club"], routes[r]["req"], routes[r]["time"])

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 20.04.6 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8375C CPU @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 211 rows, 164 columns and 911 nonzeros
Model fingerprint: 0x4f16c97f
Variable types: 1 continuous, 163 integer (78 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+04]
Presolve removed 124 rows and 31 columns
Presolve time: 0.00s
Presolved: 87 rows, 133 columns, 537 nonzeros
Variable types: 0 continuous, 133 integer (81 binary)

Root relaxation: objective 5.667780e-01, 133 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    0.56678    0   43  

Save the optimal objective value to your model in the following cell, to opt_obj_val.

In [3]:
opt_obj_val = mod.objVal

###
### CODE for Section 2
# After mod.optimize()
#if mod.status == GRB.OPTIMAL:
    # 1. Get Worst Case Seat Loss
    #worst_case_loss = z.X
    
    # 2. Calculate New Average Service Time
    # (Total service time / Total students)
   # total_new_service_time = sum(clubs[routes[r]["club"]]["students"] * routes[r]["time"] * x[r].X for r in route_ids)
   # avg_service_time = total_new_service_time / total_students
    
   # print(f"Results for beta = {beta}:")
    #print(f"  Worst-case seat loss: {worst_case_loss}")
    #print(f"  Avg service time: {avg_service_time}")
###